# A3: Edit Distance and Spelling Correction

Implement Minimum Edit Distance (Levenshtein), compute distances between misspelled and correct words, apply the Noisy Channel Model to suggest the most probable correction, and evaluate the system on a dataset.

**Dataset:** real misspellings with their corrections (codespell list, collected from Wikipedia edits) and Peter Norvig's big.txt (about 1 million words of English books) for word frequencies

In [ ]:
# nltk   : its edit_distance function, to check ours
# pandas : tables
# numpy  : the DP grid and the accuracy calculation
!pip install -q nltk pandas numpy

In [1]:
import re, urllib.request
from collections import Counter
import numpy as np      # grid for edit distance + mean for accuracy
import pandas as pd
import nltk

## 1. Minimum edit distance (Levenshtein)

Cost 1 for insert, delete and substitute. Cell D[i][j] = distance between the first i letters of the source and the first j letters of the target.

In [2]:
def edit_distance(source, target):
    n, m = len(source), len(target)

    # (n+1) x (m+1) grid of zeros; the extra row/column is the "null" (empty string) row/column
    D = np.zeros((n + 1, m + 1), dtype=int)

    D[:, 0] = np.arange(n + 1)    # left column 0,1,2,...,n -> delete all i letters (blue arrows going down)
    D[0, :] = np.arange(m + 1)    # top row    0,1,2,...,m -> insert all j letters (blue arrows going right)

    # fill the rest of the grid row by row (each cell needs its top, left and top-left neighbours)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # letters of the words are at index i-1 and j-1 because the grid has the extra null row/col
            sub = 0 if source[i-1] == target[j-1] else 1   # match is free, mismatch costs 1
            D[i, j] = min(D[i-1, j] + 1,         # from the cell above    -> delete
                          D[i, j-1] + 1,         # from the cell on left  -> insert
                          D[i-1, j-1] + sub)     # from the diagonal cell -> substitute (or match)

    return D[n, m], D     # bottom-right cell is the answer, D is the whole grid

In [3]:
dist, D = edit_distance("intention", "execution")
print("distance:", dist)
# D is a numpy array; wrap it in a DataFrame only to get the letter labels on rows/columns
pd.DataFrame(D, index=["#"] + list("intention"), columns=["#"] + list("execution"))

distance: 5


,#,e,x,e,c,u,t,i,o,n
#,0,1,2,3,4,5,6,7,8,9
i,1,1,2,3,4,5,6,6,7,8
n,2,2,2,3,4,5,6,7,7,7
t,3,3,3,3,4,5,5,6,7,8
e,4,3,4,3,4,5,6,6,7,8
n,5,4,4,4,4,5,6,7,7,7
t,6,5,5,5,5,5,5,6,7,8
i,7,6,6,6,6,6,6,5,6,7
o,8,7,7,7,7,7,7,6,5,6
n,9,8,8,8,8,8,8,7,6,5


In [4]:
# check against NLTK
for a, b in [("intention", "execution"), ("kitten", "sitting"), ("recieve", "receive")]:
    print(f"{a:<10} {b:<10} ours = {edit_distance(a, b)[0]}   nltk = {nltk.edit_distance(a, b)}")

intention  execution  ours = 5   nltk = 5
kitten     sitting    ours = 3   nltk = 3
recieve    receive    ours = 2   nltk = 2


## 2. Load the misspellings and the word frequencies

In [5]:
MISSPELL_URL = "https://raw.githubusercontent.com/codespell-project/codespell/main/codespell_lib/data/dictionary.txt"
lines = urllib.request.urlopen(MISSPELL_URL).read().decode("utf-8").splitlines()
pairs = [l.split("->") for l in lines if "->" in l]     # each line looks like  wrong->correct
pairs = [(w, c) for w, c in pairs if "," not in c and w.isalpha() and c.isalpha()]   # keep single, plain-word corrections
misspell = pd.DataFrame(pairs, columns=["wrong", "correct"])
print(len(misspell), "misspellings")
misspell.sample(5, random_state=1)

57825 misspellings


,wrong,correct
2852,akumulators,accumulators
26455,freqencies,frequencies
17814,defendent,defendant
8463,bafore,before
5554,approuve,approve


In [6]:
BIG_URL = "https://raw.githubusercontent.com/norvig/pytudes/main/data/text/big.txt"
big = urllib.request.urlopen(BIG_URL).read().decode("utf-8", errors="ignore")
counts = Counter(re.findall(r"[a-z]+", big.lower()))    # word -> how many times it appears
total = sum(counts.values())                            # total words, used later to turn counts into probabilities
print(f"{total:,} words, {len(counts):,} distinct")
counts.most_common(8)

1,105,285 words, 29,157 distinct


[('the', 80030),
 ('of', 40025),
 ('and', 38313),
 ('to', 28766),
 ('in', 22050),
 ('a', 21155),
 ('that', 12512),
 ('he', 12401)]

## 3. Distances between misspelled and correct words

In [7]:
# [0] picks only the distance (edit_distance also returns the grid)
misspell["distance"] = [edit_distance(w, c)[0] for w, c in zip(misspell["wrong"], misspell["correct"])]
misspell.sample(8, random_state=2)

,wrong,correct,distance
39204,parellelisation,parallelisation,1
45530,reputpose,repurpose,1
47907,sequemce,sequence,1
55458,unspefied,unspecified,2
49636,speficiallics,specifics,5
36120,necesities,necessities,1
41349,prepairs,prepares,2
48491,sillibus,syllabus,2


In [8]:
misspell["distance"].value_counts().sort_index().head(6)

distance
1    38525
2    16327
3     2174
4      534
5      178
6       46
Name: count, dtype: int64

## 4. Noisy channel model

Choose the correction c that maximises **P(c) x P(w | c)**:
- P(c): how common the word is (from big.txt),
- P(w | c): how likely the typo is. Simple version: each edit has probability 0.01, so P(w | c) = 0.01 ** distance.

Candidates: known words within edit distance 1 or 2 of the typo.

In [15]:
letters = "abcdefghijklmnopqrstuvwxyz"

def edits1(word):
    # every string that is exactly ONE edit away from word (most of them are not real words)
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]      # cut the word at every position
    deletes = [L + R[1:] for L, R in splits if R]                      # drop one letter
    swaps = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]  # swap two neighbours (teh -> the)
    replaces = [L + c + R[1:] for L, R in splits if R for c in letters]  # change one letter to a-z
    inserts = [L + c + R for L, R in splits for c in letters]          # add one letter a-z
    return set(deletes + swaps + replaces + inserts)

def candidates(word):
    # keep only strings that are real words (present in counts)
    d1 = {w for w in edits1(word) if w in counts}                              # real words 1 edit away
    d2 = {w2 for w1 in edits1(word) for w2 in edits1(w1) if w2 in counts}      # real words 2 edits away
    # returns (word, distance) pairs; d2 - d1 so a word reachable in 1 edit is not tagged as 2
    return [(c, 1) for c in d1] + [(c, 2) for c in d2 - d1]

def P(c):
    return counts[c] / total      # prior: how common the word is

def correct(word, p_edit=0.01):
    if word in counts:            # already a real word, leave it
        return word
    cands = candidates(word)
    if not cands:                 # nothing close enough, give the word back unchanged
        return word

    best, best_score = word, 0
    for c, d in cands:
        score = P(c) * p_edit ** d      # prior x channel
        if score > best_score:
            best, best_score = c, score
    return best

In [16]:
word = "speling"
# score every candidate: P(c) * 0.01 ** distance
table = pd.DataFrame([(c, d, counts[c], P(c) * 0.01 ** d) for c, d in candidates(word)],
                     columns=["candidate", "distance", "count", "P(c) x P(w|c)"]).sort_values("P(c) x P(w|c)", ascending=False)
print("correction:", correct(word))
table.head(8)

correction: spelling


,candidate,distance,count,P(c) x P(w|c)
0,spelling,1,4,3.618976e-08
5,feeling,2,362,3.275173e-08
21,seeing,2,207,1.872820e-08
16,speaking,2,185,1.673776e-08
13,swelling,2,167,1.510923e-08
24,smiling,2,161,1.456638e-08
11,opening,2,146,1.320926e-08
9,spring,2,70,6.333208e-09


In [17]:
for w in ["recieve", "acommodate", "teh", "occured", "wich", "thier"]:
    print(f"{w:<12} -> {correct(w)}")

recieve      -> receive
acommodate   -> accommodate
teh          -> the
occured      -> occurred
wich         -> with
thier        -> their


## 5. Evaluate on the dataset

500 misspellings whose correct word is in big.txt and that are not real words themselves. Compare three ways of choosing:
- **most frequent** candidate (ignores distance),
- **closest** candidate (ignores frequency; ties broken by frequency),
- **noisy channel** (both).

In [18]:
# test set: correct word is known, and the misspelling is not itself a real word
test = misspell[misspell["correct"].isin(counts) & ~misspell["wrong"].isin(counts)].sample(500, random_state=0)

def accuracy(fix, data):
    # fix = a correction function; fraction of misspellings it turns into the right word
    hits = [fix(w) == c for w, c in zip(data["wrong"], data["correct"])]
    return np.mean(hits)       # mean of True/False = fraction of True

def most_frequent(w):
    c = candidates(w); return max(c, key=lambda cd: counts[cd[0]])[0] if c else w          # biggest count wins
def closest(w):
    c = candidates(w); return min(c, key=lambda cd: (cd[1], -counts[cd[0]]))[0] if c else w  # smallest distance, then biggest count

results = {}
for name, f in [("most frequent", most_frequent), ("closest", closest), ("noisy channel", correct)]:
    results[name] = accuracy(f, test)
pd.Series(results, name="accuracy").round(3)

most frequent    0.544
closest          0.896
noisy channel    0.896
Name: accuracy, dtype: float64

**Effect of the channel probability.** p_edit decides how much one extra edit costs compared with word frequency.

In [19]:
rows = []
for p in [0.5, 0.1, 0.01, 0.001]:
    # same correct() each time, only the cost of an edit changes
    rows.append({"p_edit": p, "accuracy": accuracy(lambda w: correct(w, p_edit=p), test)})
pd.DataFrame(rows).round(3)

,p_edit,accuracy
0,0.500,0.602
1,0.100,0.784
2,0.010,0.896
3,0.001,0.896


In [20]:
# where the noisy channel goes wrong
test["predicted"] = [correct(w) for w in test["wrong"]]
test[test["predicted"] != test["correct"]].head(10)

,wrong,correct,distance,predicted
50517,strng,string,1,strong
48153,settins,settings,1,setting
3514,altenated,alternated,1,alienated
12241,climer,climber,1,clymer
46393,revelent,relevant,3,reverent
54549,unadvertedly,inadvertently,3,unadvertedly
18407,dependendents,dependents,3,dependendents
22143,emmision,emission,2,excision
33954,manal,manual,1,canal
40640,polution,pollution,1,solution


## Conclusion
- The edit distance table gives the minimum number of insertions, deletions and substitutions; ours matches NLTK.
- Most real misspellings are 1 or 2 edits away from the correct word.
- Frequency alone is a poor guide; distance matters most. The noisy channel with a small p_edit behaves almost like "closest, ties by frequency", and a large p_edit lets frequent but distant words win.
- A better channel model would learn which edits are common (e/a confusions, doubled letters) from training pairs instead of treating every edit the same.
- Errors come from typos further than 2 edits, and from frequent real words that are just as close (strng -> strong, altenated -> alienated).